# Shipping buffer target (run 1)

Time series of the DBR **shipping buffer target** (`shippingBufferTarget`) for a single simulation experiment run: one combined plot for all products, then one figure per product.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml

from manusim.metrics import ExperimentMetrics

In [ ]:
# --- configure experiment ---
EXPERIMENT_PATH = Path(
    "C:/github/master/manufacturing-scheduling/data/experiments/simulation/2606252207_dbr_sb_adjust"
)
RUN = 1

experiment_path = EXPERIMENT_PATH.resolve()
if not experiment_path.exists():
    raise FileNotFoundError(f"Experiment folder not found: {experiment_path}")
print(experiment_path)

In [ ]:
metrics = ExperimentMetrics(experiment_path)
metrics.read_logs(metrics=["shippingBufferTarget"])

logs = metrics.logs.copy()
logs = logs.loc[logs["run"] == RUN].copy()
logs = logs.sort_values(["key", "time"])

sb = logs.loc[logs["variable"] == "shippingBufferTarget"].copy()
products = sorted(sb["key"].dropna().unique())

print(f"Log rows (run {RUN}): {len(sb):,}")
print(f"Products: {products}")
if sb.empty:
    print(
        "WARNING: no shippingBufferTarget logs found. "
        "Check experiment.save_logs in config and that RUN exists."
    )

In [ ]:
config_path = experiment_path / ".hydra" / "config.yaml"
if config_path.exists():
    config = yaml.safe_load(config_path.read_text(encoding="utf-8"))
    sim_cfg = config.get("simulation", {})
    print(
        "Buffer controls:",
        {
            "sb_update": sim_cfg.get("sb_update"),
            "buffers_update_warmup": sim_cfg.get("buffers_update_warmup"),
            "buffers_update_multiplyer": sim_cfg.get("buffers_update_multiplyer"),
        },
    )
    initial_targets = {
        p: cfg.get("shipping_buffer")
        for p, cfg in config.get("products", {}).items()
    }
    print("Initial shipping_buffer from config:")
    for product, value in sorted(initial_targets.items()):
        print(f"  {product}: {value}")

## All products

In [ ]:
plt.figure(figsize=(12, 7))
for product in products:
    product_df = sb.loc[sb["key"] == product].sort_values("time")
    if product_df.empty:
        continue
    plt.plot(product_df["time"], product_df["value"], label=product)

plt.title(f"Shipping buffer target — all products (run {RUN})")
plt.xlabel("simulation time")
plt.ylabel("target (hours)")
plt.legend(title="product", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
sb_stats = (
    sb.loc[sb['time'] >= 4000000]
    .groupby('key')
    .agg(
        mean_value=('value', 'mean'),
        max_value=('value', 'max'),
        min_value=('value', 'min'),
        last_value=('value', lambda x: x.iloc[-1]),
    )
)
sb_stats

## Per product

In [ ]:
for product in products:
    product_df = sb.loc[sb["key"] == product].sort_values("time")
    if product_df.empty:
        continue

    plt.figure(figsize=(10, 6))
    plt.plot(product_df["time"], product_df["value"])
    plt.title(f"Shipping buffer target — {product} (run {RUN})")
    plt.xlabel("simulation time")
    plt.ylabel("target (hours)")
    plt.tight_layout()
    plt.show()